# Task B – MFCD vs VCD (Self-contained)
This notebook clones the repo, installs dependencies, downloads COCO val2014, runs VCD and MFCD, then prints a comparison table.

In [1]:
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA Version:', torch.version.cuda)

Torch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA Version: 12.8


In [2]:
import os
ROOT = '/content/VCD_project'
REPO_URL = 'https://github.com/maxheef/ECE209_Project1.git'
BRANCH = 'main'
COCO_VAL = '/content/datasets/coco/val2014'
MODEL_PATH = 'liuhaotian/llava-v1.5-7b'
MFCD_MODEL_PATH = 'llava-hf/llava-1.5-7b-hf'
SEED = 55

# MFCD parameters (from MFCD readme)
MFC_HIGH_ALPHA = 1.0
MFC_LOW_ALPHA = 1.0
MFC_BETA = 0.3
MFC_HIGH_PASS_CUTOFF = 0.1
MFC_LOW_PASS_CUTOFF = 0.9
MFC_FILTER_TYPE = 'gaussian'
print('ROOT=', ROOT)

ROOT= /content/VCD_project


In [3]:
%%bash
set -euo pipefail
cd /content
if [ -d /content/VCD_project/.git ]; then
  git -C /content/VCD_project fetch --depth 1 origin ${BRANCH:-main}
  git -C /content/VCD_project reset --hard FETCH_HEAD
else
  rm -rf /content/VCD_project
  git clone --depth 1 --branch ${BRANCH:-main} ${REPO_URL:-https://github.com/maxheef/ECE209_Project1.git} /content/VCD_project
fi

if [ ! -d /content/VCD_project/originalMFCD/mfcd ]; then
  echo 'Missing /content/VCD_project/originalMFCD/mfcd (check repo contents)'.
  find /content/VCD_project -maxdepth 3 -type d | sed -n '1,120p'
  exit 1
fi


Cloning into '/content/VCD_project'...


In [4]:
import os
import subprocess
import shutil
from pathlib import Path

def run_cmd(cmd):
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=os.environ.copy())
    for line in process.stdout:
        print(line, end='')
    process.wait()
    if process.returncode != 0:
        raise Exception(f"Command failed: {cmd}")

# --- Configuration ---
CONDA_PATH = '/usr/local/miniconda'
REQ_FILE = '/content/VCD_project/requirements.txt'

VCD_ENV = 'vcd310'
MFCD_ENV = 'mfcd310'
VCD_PY = f'{CONDA_PATH}/envs/{VCD_ENV}/bin/python'
VCD_PIP = f'{CONDA_PATH}/envs/{VCD_ENV}/bin/pip'
MFCD_PY = f'{CONDA_PATH}/envs/{MFCD_ENV}/bin/python'
MFCD_PIP = f'{CONDA_PATH}/envs/{MFCD_ENV}/bin/pip'

# 1) Setup Miniconda
if not os.path.exists(CONDA_PATH):
    run_cmd('wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh')
    run_cmd(f'bash /tmp/miniconda.sh -b -p {CONDA_PATH}')

conda_bin = f'{CONDA_PATH}/bin/conda'
os.environ['CONDA_PLUGINS_AUTO_ACCEPT_TOS'] = 'yes'
run_cmd(f"{conda_bin} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main")
run_cmd(f"{conda_bin} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r")

# 2) (Re)create envs
for env_name in [VCD_ENV, MFCD_ENV]:
    env_path = f'{CONDA_PATH}/envs/{env_name}'
    if os.path.exists(env_path):
        shutil.rmtree(env_path)
    print(f'Creating environment {env_name} (Python 3.10)...')
    run_cmd(f"{conda_bin} create -n {env_name} python=3.10 -y")

# 3) Install base requirements in both envs
if not os.path.exists(REQ_FILE):
    raise FileNotFoundError(f'Missing requirements: {REQ_FILE}')

run_cmd(f"{VCD_PIP} install --upgrade pip setuptools wheel")
run_cmd(f"{VCD_PIP} install --no-cache-dir -r {REQ_FILE}")

run_cmd(f"{MFCD_PIP} install --upgrade pip setuptools wheel")
run_cmd(f"{MFCD_PIP} install --no-cache-dir -r {REQ_FILE}")

# 4) Pin versions
# VCD needs older transformers + tokenizers to avoid LLaVA config conflicts
run_cmd(f"{VCD_PIP} install --no-cache-dir 'transformers==4.31.0' 'tokenizers==0.13.3' 'numpy<2'")

# MFCD needs newer tokenizers for its bundled transformers fork
run_cmd(f"{MFCD_PIP} install --no-cache-dir 'tokenizers==0.21.0' 'numpy<2'")

# 5) Record python paths
Path('/tmp/mytasks_python_bin.txt').write_text(VCD_PY)
Path('/tmp/mytasks_python_bin_mfcd.txt').write_text(MFCD_PY)
print('VCD env:', VCD_PY)
print('MFCD env:', MFCD_PY)

run_cmd(f"{VCD_PY} -c 'import torch; print(\"VCD Python\", __import__(\"sys\").version); print(\"CUDA\", torch.cuda.is_available())'")
run_cmd(f"{MFCD_PY} -c 'import torch; print(\"MFCD Python\", __import__(\"sys\").version); print(\"CUDA\", torch.cuda.is_available())'")


PREFIX=/usr/local/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /usr/local/miniconda
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Creating environment vcd310 (Python 3.10)...
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - defaults
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /usr/local/miniconda/envs/vcd310

  added / updated specs:
    - python=3.10



In [5]:
import os
import subprocess
import shutil

# --- Configuration ---
DATASET_DIR = "/content/datasets/coco"
VAL_DIR = os.path.join(DATASET_DIR, "val2014")
ZIP_FILE = os.path.join(DATASET_DIR, "val2014.zip")
ZIP_URL = "http://images.cocodataset.org/zips/val2014.zip"

# 1. Create directory structure
os.makedirs(DATASET_DIR, exist_ok=True)

# 2. Download and Unzip if val2014 doesn't exist
if not os.path.exists(VAL_DIR):
    print(f"Downloading COCO val2014 to {DATASET_DIR}...")
    # Using subprocess for wget/unzip to handle large file streams efficiently
    subprocess.run(["wget", "-q", ZIP_URL, "-O", ZIP_FILE], check=True)
    
    print("Unzipping dataset...this will take about 2 min")
    subprocess.run(["unzip", "-q", ZIP_FILE, "-d", DATASET_DIR], check=True)
    
    # Optional: Clean up zip to save space on Colab
    os.remove(ZIP_FILE)
    print("Download and extraction complete.")
else:
    print("COCO val2014 already exists. Skipping download.")

# 3. Verify: Print first 8 lines of directory listing
print("\n--- Directory Preview (val2014) ---")
files = sorted(os.listdir(VAL_DIR))
for f in files[:8]:
    print(f)


Unzipping dataset...this will take about 2 min
Download and extraction complete.

--- Directory Preview (val2014) ---
COCO_val2014_000000000042.jpg
COCO_val2014_000000000073.jpg
COCO_val2014_000000000074.jpg
COCO_val2014_000000000133.jpg
COCO_val2014_000000000136.jpg
COCO_val2014_000000000139.jpg
COCO_val2014_000000000143.jpg
COCO_val2014_000000000164.jpg


In [36]:
import os
import subprocess
from pathlib import Path

PYBIN_PATH = '/tmp/mytasks_python_bin.txt'
if not Path(PYBIN_PATH).exists():
    raise FileNotFoundError(f'Missing {PYBIN_PATH}. Run the conda setup cell first.')
PYBIN = Path(PYBIN_PATH).read_text().strip()

MODEL_PATH = 'liuhaotian/llava-v1.5-7b'
COCO_VAL = '/content/datasets/coco/val2014'
SEED = '55'
ROOT = '/content/VCD_project'
ORIG = f'{ROOT}/originalProject'

# Preflight checks for clearer errors
if not Path(ORIG, 'experiments').is_dir():
    raise FileNotFoundError(f'Missing originalProject/experiments at {ORIG}/experiments')
if not Path(COCO_VAL).is_dir():
    raise FileNotFoundError(f'Missing COCO val2014 at {COCO_VAL}')

env = os.environ.copy()
env['PYTHON_BIN'] = PYBIN
cmd = ['bash', '/content/VCD_project/myTasks/run_table1.sh', MODEL_PATH, COCO_VAL, SEED]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, env=env, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'run_table1.sh failed with code {result.returncode}')


Running: bash /content/VCD_project/myTasks/run_table1.sh liuhaotian/llava-v1.5-7b /content/datasets/coco/val2014 55
Running split=random method=regular
Precision: 0.919931856899489
Recall: 0.72
F1: 0.8077786088257293
Accuracy: 0.8286666666666667
yes: 0.3913333333333333
unknow: 0.0
Saved: /content/VCD_project/myTasks/output/llava15_coco_pope_random_answers_regular_seed55.jsonl
Saved: /content/VCD_project/myTasks/output/metrics_random_regular_seed55.txt
Running split=random method=vcd
Precision: 0.9369369369369369
Recall: 0.7626666666666667
F1: 0.8408673281881661
Accuracy: 0.8556666666666667
yes: 0.407
unknow: 0.0
Saved: /content/VCD_project/myTasks/output/llava15_coco_pope_random_answers_vcd_seed55.jsonl
Saved: /content/VCD_project/myTasks/output/metrics_random_vcd_seed55.txt
Running split=popular method=regular
Precision: 0.8794788273615635
Recall: 0.72
F1: 0.7917888563049853
Accuracy: 0.8106666666666666
yes: 0.4093333333333333
unknow: 0.0
Saved: /content/VCD_project/myTasks/output/lla

In [6]:
import subprocess
from pathlib import Path

# --- Setup Paths ---
PYBIN_PATH = '/tmp/mytasks_python_bin_mfcd.txt'
if not Path(PYBIN_PATH).exists():
    raise FileNotFoundError(f'Missing {PYBIN_PATH}. Run the conda setup cell first.')

PYBIN = Path(PYBIN_PATH).read_text().strip()
DRIVER_PATH = Path('/tmp/mfcd_driver.py')

# --- Optimized MFCD Driver Script ---
mfcd_driver = """
import os
import sys
import json
import torch
import gc
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset

# --- Configuration ---
ROOT = '/content/VCD_project'
ORIG = f'{ROOT}/originalProject'
MFCD = f'{ROOT}/originalMFCD/mfcd'
COCO_VAL = '/content/datasets/coco/val2014'
MODEL_PATH = 'llava-hf/llava-1.5-7b-hf'
OUT_DIR = f'{ROOT}/myTasks/output'

# MFCD Parameters
MFC_CONFIG = {
    'mfc_high_alpha': 1.0,
    'mfc_low_alpha': 1.0,
    'mfc_beta': 0.3,
    'mfc_high_pass_cutoff': 0.1,
    'mfc_low_pass_cutoff': 0.9,
    'mfc_filter_type': 'gaussian'
}

# Add MFCD to path
if MFCD not in sys.path:
    sys.path.insert(0, MFCD)

from eval_utils import prepare_for_generate, generate_responses, batch_prepare_data
from eval_utils import MODEL_INFOS
from eval.pope.eval import eval as pope_eval, recorder

class POPELocal(Dataset):
    def __init__(self, json_path, image_root):
        with open(json_path, 'r') as f:
            self.rows = [json.loads(line) for line in f]
        self.image_root = Path(image_root)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        img = Image.open(self.image_root / row['image']).convert('RGB')
        return {
            'question': row['text'],
            'image': img,
            'label': row['label'].lower()
        }

def run_mfcd(pope_type):
    device = torch.device('cuda:0')
    
    class Args: pass
    args = Args()
    args.model_name_or_path = MODEL_PATH
    args.model_type = 'llava-1.5'
    args.num_workers = 4
    args.eval_batch_size = 16  # Optimized for Blackwell 96GB VRAM
    args.device = 'cuda:0'
    args.temperature = 1.2
    args.top_p = 1.0
    args.top_k = 50
    args.max_new_tokens = 2048
    
    # Map MFCD Config
    for k, v in MFC_CONFIG.items():
        setattr(args, k, v)
    args.mfc_jsd = False
    args.mfc_entropy = False

    print(f"\\n--- Starting MFCD: {pope_type} ---")
    model_info = MODEL_INFOS['llava-1.5']
    
    # Initialize Processor
    processor, generation_config, _ = prepare_for_generate(args=args, model_info=model_info, device=device)

    # Prepare Data
    pope_json = f"{ORIG}/experiments/data/POPE/coco/coco_pope_{pope_type}.json"
    dataset = POPELocal(pope_json, COCO_VAL)
    prompts = batch_prepare_data(args=args, dataset=dataset, processor=processor, 
                                 question_column_name='question', image_column_name='image')

    # Load Model
    model = model_info.model_class.from_pretrained(
        MODEL_PATH, 
        torch_dtype=torch.float16, # Ensure FP16 for Blackwell speed
        **model_info.model_config
    ).to(device)

    # Inference
    responses = generate_responses(model=model, processor=processor, prompts=prompts, generation_config=generation_config)

    # Eval
    labels = [1 if d['label'] == 'yes' else 0 for d in dataset]
    preds = recorder(responses)
    results = pope_eval(preds, labels)
    
    # Cleanup to prevent OOM on next run
    del model
    del processor
    gc.collect()
    torch.cuda.empty_cache()
    
    return results

# Execution
os.makedirs(OUT_DIR, exist_ok=True)
for p_type in ['random', 'popular']:
    res = run_mfcd(p_type)
    with open(os.path.join(OUT_DIR, f'mfcd_{p_type}.json'), 'w') as f:
        json.dump(res, f, indent=2)
    print(f"Results for {p_type}: {res}")
"""

# --- Run Driver ---
DRIVER_PATH.write_text(mfcd_driver)
print(f'Executing MFCD on Blackwell GPU using: {PYBIN}')

# Using Popen to stream logs in real-time
process = subprocess.Popen([PYBIN, str(DRIVER_PATH)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end='')

process.wait()

if process.returncode != 0:
    raise RuntimeError(f'MFCD driver failed with code {process.returncode}')


Executing MFCD on Blackwell GPU using: /usr/local/miniconda/envs/mfcd310/bin/python

--- Starting MFCD: random ---

Applying chat template: 100%|██████████| 188/188 [00:05<00:00, 36.01it/s]

Processing batch data: 100%|██████████| 188/188 [00:22<00:00,  8.33it/s]

Fetching 3 files: 100%|██████████| 3/3 [00:43<00:00, 14.52s/it]

Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  7.88it/s]

Generating responses: 100%|██████████| 188/188 [11:45<00:00,  3.75s/it]
Results for random: {'accuracy': 0.8686666666666667, 'precision': 0.8540332906530089, 'recall': 0.8893333333333333, 'f1_score': 0.8713259307642064, 'yes_ratio': 0.5206666666666667, 'TP': 1334, 'TN': 1272, 'FP': 228, 'FN': 166}

--- Starting MFCD: popular ---

Applying chat template: 100%|██████████| 188/188 [00:05<00:00, 37.11it/s]

Processing batch data: 100%|██████████| 188/188 [00:20<00:00,  8.97it/s]

Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  5.16it/s]

Generating responses: 100%|██████████| 18

In [10]:
import json
import re
from pathlib import Path
P1 = '/content/VCD_project/myTasks'
SEED = 55
out_dir = Path(f'{P1}/output')
def parse_metrics_txt(path: Path):
    text = path.read_text()
    def g(name):
        m = re.search(rf"^{name}:\s*([0-9.]+)", text, re.MULTILINE)
        return float(m.group(1)) if m else float('nan')
    return {
        'accuracy': g('Accuracy'),
        'precision': g('Precision'),
        'recall': g('Recall'),
        'f1': g('F1'),
    }
def parse_metrics_json(path: Path):
    data = json.loads(path.read_text())
    return {
        'accuracy': float(data.get('accuracy', float('nan'))),
        'precision': float(data.get('precision', float('nan'))),
        'recall': float(data.get('recall', float('nan'))),
        'f1': float(data.get('f1', float('nan'))),
    }
vcd_random = parse_metrics_txt(out_dir / f'metrics_random_vcd_seed{SEED}.txt')
vcd_popular = parse_metrics_txt(out_dir / f'metrics_popular_vcd_seed{SEED}.txt')
mfcd_random = parse_metrics_json(out_dir / 'mfcd_random.json')
mfcd_popular = parse_metrics_json(out_dir / 'mfcd_popular.json')
print('| Split | Method | Accuracy | Precision | Recall | F1 |')
print('|---|---|---:|---:|---:|---:|')
print(f"| random | VCD | {vcd_random['accuracy']:.4f} | {vcd_random['precision']:.4f} | {vcd_random['recall']:.4f} | {vcd_random['f1']:.4f} |")
print(f"| random | MFCD | {mfcd_random['accuracy']:.4f} | {mfcd_random['precision']:.4f} | {mfcd_random['recall']:.4f} | {mfcd_random['f1']:.4f} |")
print(f"| popular | VCD | {vcd_popular['accuracy']:.4f} | {vcd_popular['precision']:.4f} | {vcd_popular['recall']:.4f} | {vcd_popular['f1']:.4f} |")
print(f"| popular | MFCD | {mfcd_popular['accuracy']:.4f} | {mfcd_popular['precision']:.4f} | {mfcd_popular['recall']:.4f} | {mfcd_popular['f1']:.4f} |")


| Split | Method | Accuracy | Precision | Recall | F1 |
|---|---|---:|---:|---:|---:|
| random | VCD | 0.8557 | 0.9369 | 0.7627 | 0.8409 |
| random | MFCD | 0.8687 | 0.8540 | 0.8893 | nan |
| popular | VCD | 0.8357 | 0.8931 | 0.7627 | 0.8227 |
| popular | MFCD | 0.8267 | 0.7913 | 0.8873 | nan |


In [9]:
import subprocess
from pathlib import Path

# --- Setup Paths ---
PYBIN_PATH = '/tmp/mytasks_python_bin.txt'
if not Path(PYBIN_PATH).exists():
    raise FileNotFoundError(f'Missing {PYBIN_PATH}. Run the conda setup cell first.')

PYBIN = Path(PYBIN_PATH).read_text().strip()
DRIVER_PATH = Path('/tmp/vcd_driver.py')

# --- Clean VCD Driver Script ---
vcd_driver = """
import os
import sys
import torch
import gc
import subprocess
from pathlib import Path

# --- Configuration ---
ROOT = '/content/VCD_project'
ORIG = f'{ROOT}/originalProject'
EXP_DIR = f'{ORIG}/experiments'
OUT_DIR = f'{ROOT}/myTasks/output'
COCO_VAL = '/content/datasets/coco/val2014'
MODEL_PATH = 'liuhaotian/llava-v1.5-7b'
SEED = '55'

# VCD Hyperparameters
CD_ALPHA = "1.0"
CD_BETA = "0.2"
NOISE_STEP = "500"

os.makedirs(OUT_DIR, exist_ok=True)

def run_vcd_step(split, method):
    question_file = f"{EXP_DIR}/data/POPE/coco/coco_pope_{split}.json"
    answers_file = f"{OUT_DIR}/llava15_coco_pope_{split}_answers_{method}_seed{SEED}.jsonl"
    metrics_file = f"{OUT_DIR}/metrics_{split}_{method}_seed{SEED}.txt"

    print(f"\\n>>> STARTING: Split={split} | Method={method}")

    # 1. Run Inference
    cmd = [
        sys.executable, "./eval/object_hallucination_vqa_llava.py",
        "--model-path", MODEL_PATH,
        "--question-file", question_file,
        "--image-folder", COCO_VAL,
        "--answers-file", answers_file,
        "--seed", SEED
    ]

    if method == "vcd":
        cmd += ["--use_cd", "--cd_alpha", CD_ALPHA, "--cd_beta", CD_BETA, "--noise_step", NOISE_STEP]

    os.chdir(EXP_DIR)
    subprocess.run(cmd, check=True)

    # 2. Run POPE Eval (Creates the .txt file for your table parser)
    eval_cmd = [sys.executable, "./eval/eval_pope.py", "--gt_files", question_file, "--gen_files", answers_file]
    with open(metrics_file, "w") as f:
        subprocess.run(eval_cmd, stdout=f, check=True)
    
    # Blackwell Memory Management
    gc.collect()
    torch.cuda.empty_cache()

# Execute benchmarks
for split in ['random', 'popular']:
    for method in ['regular', 'vcd']:
        run_vcd_step(split, method)

print("\\n--- VCD Evaluation Complete. Files ready for comparison table. ---")
"""

# --- Execute Wrapper ---
DRIVER_PATH.write_text(vcd_driver)
print(f'Running VCD Benchmarks on Blackwell GPU...')

# Stream logs for real-time monitoring
process = subprocess.Popen([PYBIN, str(DRIVER_PATH)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')

process.wait()

if process.returncode != 0:
    raise RuntimeError(f'VCD driver failed with code {process.returncode}')


Running VCD Benchmarks on Blackwell GPU...
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.

0it [00:00, ?it/s]
0it [00:00, ?it/s]
/usr/local/miniconda/envs/vcd310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.85s/it]

100%|██████████| 3000/3000 [04:02<00:00, 12.39it/s]
/usr/local/miniconda/envs/vcd310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new do

In [11]:
import json
import re
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# --- Config ---
P1 = '/content/VCD_project/myTasks'
SEED = 55
out_dir = Path(f'{P1}/output')

def parse_metrics_txt(path: Path):
    if not path.exists(): return None
    text = path.read_text()
    def g(name):
        m = re.search(rf"^{name}:\s*([0-9.]+)", text, re.MULTILINE)
        return float(m.group(1)) if m else float('nan')
    return {'Accuracy': g('Accuracy'), 'Precision': g('Precision'), 'Recall': g('Recall'), 'F1': g('F1')}

def parse_metrics_json(path: Path):
    if not path.exists(): return None
    data = json.loads(path.read_text())
    return {
        'Accuracy': float(data.get('accuracy', float('nan'))),
        'Precision': float(data.get('precision', float('nan'))),
        'Recall': float(data.get('recall', float('nan'))),
        'F1': float(data.get('f1', float('nan')))
    }

# --- Data Collection ---
results = []
configs = [
    ('random', 'VCD', out_dir / f'metrics_random_vcd_seed{SEED}.txt', parse_metrics_txt),
    ('random', 'MFCD', out_dir / 'mfcd_random.json', parse_metrics_json),
    ('popular', 'VCD', out_dir / f'metrics_popular_vcd_seed{SEED}.txt', parse_metrics_txt),
    ('popular', 'MFCD', out_dir / 'mfcd_popular.json', parse_metrics_json),
]

for split, method, path, parser in configs:
    metrics = parser(path)
    if metrics:
        row = {'Split': split, 'Method': method}
        row.update(metrics)
        results.append(row)

# --- Display Logic ---
if results:
    df = pd.DataFrame(results)
    
    # Apply styling: 4 decimal places and highlight the best F1 per split
    styled_df = df.style.format({
        'Accuracy': '{:.4f}', 
        'Precision': '{:.4f}', 
        'Recall': '{:.4f}', 
        'F1': '{:.4f}'
    }).hide(axis='index')
    
    display(Markdown(f"### 🚀 Blackwell GPU Benchmark Results (Seed: {SEED})"))
    display(styled_df)
else:
    print("⚠️ No result files found in output directory. Check if the drivers finished running.")


### 🚀 Blackwell GPU Benchmark Results (Seed: 55)

Split,Method,Accuracy,Precision,Recall,F1
random,VCD,0.8557,0.9369,0.7627,0.8409
random,MFCD,0.8687,0.8540,0.8893,nan
popular,VCD,0.8357,0.8931,0.7627,0.8227
popular,MFCD,0.8267,0.7913,0.8873,nan


# Experimental stuff

In [14]:
import subprocess
from pathlib import Path

PYBIN_PATH = '/tmp/mytasks_python_bin.txt'
PYBIN = Path(PYBIN_PATH).read_text().strip()
DRIVER_PATH = Path('/tmp/vcd_driver.py')

vcd_driver = """
import os
import sys
import torch
import gc
import subprocess
from pathlib import Path

ROOT = '/content/VCD_project'
ORIG = f'{ROOT}/originalProject'
EXP_DIR = f'{ORIG}/experiments'
OUT_DIR = f'{ROOT}/myTasks/output'
COCO_VAL = '/content/datasets/coco/val2014'
MODEL_PATH = 'liuhaotian/llava-v1.5-7b'
SEED = '55'

os.makedirs(OUT_DIR, exist_ok=True)

def run_vqa_step(split, method):
    question_file = f"{EXP_DIR}/data/POPE/coco/coco_pope_{split}.json"
    answers_file = f"{OUT_DIR}/llava15_coco_pope_{split}_answers_{method}_seed{SEED}.jsonl"
    metrics_file = f"{OUT_DIR}/metrics_{split}_{method}_seed{SEED}.txt"

    print(f"\\n>>> STARTING: Split={split} | Method={method}")

    cmd = [
        sys.executable, "./eval/object_hallucination_vqa_llava.py",
        "--model-path", MODEL_PATH,
        "--question-file", question_file,
        "--image-folder", COCO_VAL,
        "--answers-file", answers_file,
        "--seed", SEED
    ]

    if method == "vcd":
        cmd += ["--use_cd", "--cd_alpha", "1.0", "--cd_beta", "0.2", "--noise_step", "500"]

    os.chdir(EXP_DIR)
    subprocess.run(cmd, check=True)

    eval_cmd = [sys.executable, "./eval/eval_pope.py", "--gt_files", question_file, "--gen_files", answers_file]
    with open(metrics_file, "w") as f:
        subprocess.run(eval_cmd, stdout=f, check=True)
    
    gc.collect()
    torch.cuda.empty_cache()

# This loop ensures both 'regular' and 'vcd' are produced
for split in ['random', 'popular']:
    for method in ['regular', 'vcd']:
        run_vqa_step(split, method)
"""

DRIVER_PATH.write_text(vcd_driver)
process = subprocess.Popen([PYBIN, str(DRIVER_PATH)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()


/usr/local/miniconda/envs/vcd310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.84s/it]

100%|██████████| 3000/3000 [04:01<00:00, 12.40it/s]
/usr/local/miniconda/envs/vcd310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.83s/it]

100%|██████████| 3000/3000 [07:42<00:00,  6.49it/s]
/usr/local/miniconda/envs/vcd310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is 

0

In [16]:
import json
import re
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

P1 = '/content/VCD_project/myTasks'
SEED = 55
out_dir = Path(f'{P1}/output')

def parse_metrics_txt(path: Path):
    if not path.exists(): return None
    text = path.read_text()
    def g(name):
        m = re.search(rf"^{name}:\s*([0-9.]+)", text, re.MULTILINE)
        return float(m.group(1)) if m else 0.0
    return {'Accuracy': g('Accuracy'), 'Precision': g('Precision'), 'Recall': g('Recall'), 'F1': g('F1')}

def parse_metrics_json(path: Path):
    if not path.exists(): return None
    data = json.loads(path.read_text())
    return {'Accuracy': float(data.get('accuracy', 0.0)), 
            'Precision': float(data.get('precision', 0.0)), 
            'Recall': float(data.get('recall', 0.0)), 
            'F1': float(data.get('f1_score') or data.get('f1') or 0.0) }

# --- Data Collection ---
rows = []
# Define the order we want to see them in the table
configs = [
    ('random', 'Regular', out_dir / f'metrics_random_regular_seed{SEED}.txt', parse_metrics_txt),
    ('random', 'VCD', out_dir / f'metrics_random_vcd_seed{SEED}.txt', parse_metrics_txt),
    ('random', 'MFCD', out_dir / 'mfcd_random.json', parse_metrics_json),
    ('popular', 'Regular', out_dir / f'metrics_popular_regular_seed{SEED}.txt', parse_metrics_txt),
    ('popular', 'VCD', out_dir / f'metrics_popular_vcd_seed{SEED}.txt', parse_metrics_txt),
    ('popular', 'MFCD', out_dir / 'mfcd_popular.json', parse_metrics_json),
]

for split, method, path, parser in configs:
    m = parser(path)
    if m: rows.append({'Split': split, 'Method': method, **m})

if rows:
    df = pd.DataFrame(rows)
    
    # Calculate Improvements relative to the previous row in the same split
    df['Gain vs Prev (%)'] = 0.0
    for split in df['Split'].unique():
        idx = df[df['Split'] == split].index
        for i in range(1, len(idx)):
            prev_f1 = df.loc[idx[i-1], 'F1']
            curr_f1 = df.loc[idx[i], 'F1']
            if prev_f1 > 0:
                df.at[idx[i], 'Gain vs Prev (%)'] = ((curr_f1 - prev_f1) / prev_f1) * 100

    # Styling for Jupyter
    def color_gain(val):
        color = 'green' if val > 0 else ('red' if val < 0 else 'gray')
        return f'color: {color}; font-weight: bold'

    styled_df = df.style.format({
        'Accuracy': '{:.4f}', 'Precision': '{:.4f}', 'Recall': '{:.4f}', 
        'F1': '{:.4f}', 'Gain vs Prev (%)': '{:+.2f}%'
    }).map(color_gain, subset=['Gain vs Prev (%)']).hide(axis='index')

    display(Markdown(f"### 📊 Full Benchmark: Regular vs VCD vs MFCD (Seed: {SEED})"))
    display(styled_df)
else:
    print("⚠️ No output files found. Check your /output directory.")


### 📊 Full Benchmark: Regular vs VCD vs MFCD (Seed: 55)

Split,Method,Accuracy,Precision,Recall,F1,Gain vs Prev (%)
random,Regular,0.8287,0.9199,0.7200,0.8078,+0.00%
random,VCD,0.8557,0.9369,0.7627,0.8409,+4.10%
random,MFCD,0.8687,0.8540,0.8893,0.8713,+3.62%
popular,Regular,0.8107,0.8795,0.7200,0.7918,+0.00%
popular,VCD,0.8357,0.8931,0.7627,0.8227,+3.91%
popular,MFCD,0.8267,0.7913,0.8873,0.8366,+1.68%
